In [ ]:
import numpyro
import numpyro.distributions as dist
import arviz as az

numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp
import optimistix as optx

# jax.config.update("jax_disable_jit", True)
jax.local_device_count()

In [ ]:
def reactor_MB(y, params):
    R = 8.314  # J/(mol*K)
    T_ref = 273.15 + 50  # K
    A1, Ea1, A2, Ea2, A3, Ea3, T, p, tau = params

    k1 = A1 * jnp.exp(-Ea1 / R * (1 / T_ref - 1 / T))
    k2 = A2 * jnp.exp(-Ea2 / R * (1 / T_ref - 1 / T))
    k3 = A3 * jnp.exp(-Ea3 / R * (1 / T_ref - 1 / T))

    r1 = k1 * y[0] * p
    r2 = k2 * y[1] * p
    r3 = k3 * y[0] * p**2

    c_in = jnp.array([1.0, 0.0, 0.0])  # mol/s
    dy_dt = jnp.array(
        [
            (c_in[0] - y[0]) / tau - r1 - r3,
            (c_in[1] - y[1]) / tau + r1 - r2,
            (c_in[2] - y[2]) / tau + r2 + r3,
        ]
    )
    return dy_dt


def reactor_model(A1, Ea1, A2, Ea2, A3, Ea3, T, p_i, tau):
    solver = optx.Newton(rtol=1e-5, atol=1e-8)
    sol = optx.root_find(
        fn=reactor_MB,
        solver=solver,
        y0=jnp.array([0.5, 0.0, 0.0]),
        args=(A1, Ea1, A2, Ea2, A3, Ea3, T, p_i, tau),
    )
    c = sol.value
    return c[1] / tau

In [ ]:
TRUE_PARAMS = {
    "A1": 1.0,
    "Ea1": 50000.0,
    "A2": 0.5,
    "Ea2": 60000.0,
    "A3": 0.1,
    "Ea3": 70000.0,
}